### **Semana 9. Workflows, agentes, ReAct y criterios de parada**

**Pregunta central:** ¿cuándo una política adaptativa mejora el compromiso entre recuperación, volumen de contexto y acciones, y qué evidencia adicional hace falta para llamar *agente* a un controlador?

Este cuaderno trabaja sobre **todo SQuAD 1.1 dev**, con 48 artículos, 2.067 pasajes originales y 10.570 preguntas escritas por personas. No contiene consultas inventadas ni resultados precargados. Separa el benchmark BM25 **sin LLM** de un piloto opcional con Qwen auténtico.

**Tiempo docente:** teoría 2 horas; el laboratorio asociado, 4 horas.

#### **1. Protocolo antes de observar métricas**

1. A0, workflow fijo, una búsqueda con top-1.
2. A1, workflow fijo, una búsqueda con top-3.
3. B, controlador adaptativo: top-1, ampliación a top-3 si cobertura léxica baja.
4. C, opcional, agente Qwen que elige acciones y respuestas.

A0, A1 y B usan el **mismo BM25** sobre todos los documentos. B se calibra únicamente con preguntas de 12 artículos; los 36 restantes son para evaluar. Las respuestas humanas se guardan fuera del índice.

**Hipótesis no trivial:** B puede ahorrar contexto respecto a A1, pero puede perder evidencia y ejecutar dos búsquedas. Se mide el compromiso, no se garantiza una victoria.

#### **2. Archivos originales y comprobación de integridad**

In [ ]:
from pathlib import Path
import json, sys
root = Path.cwd().resolve()
for parent in (root, *root.parents):
    if (parent / "Semana9" / "agent_benchmark.py").is_file():
        root = parent
        break
else:
    raise FileNotFoundError("Abrir JupyterLab dentro de CC-0F4")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from Semana9.prepare_real_data import verify, EXPECTED_PASSAGES, EXPECTED_QUESTIONS
from Semana9.agent_benchmark import (
    BM25Tool, real_paths, jsonl, lexical_coverage, tokens, evaluate_run,
    run_workflow, run_adaptive,
)
manifest = verify(root / "Semana9" / "datos_reales")
assert manifest["queries"] == EXPECTED_QUESTIONS == 10570
assert manifest["passages"] == EXPECTED_PASSAGES == 2067
print("Origen:", manifest["dataset_source"])
print("SHA256:", manifest["source_sha256"])
print("Partición:", manifest["split_queries"])

#### **3. Consulta y documento auténticos**

Se conserva la pregunta con ID de SQuAD original, su título de Wikipedia y el pasaje de origen. El pasaje es una etiqueta de **origen de respuesta**, no un juicio exhaustivo de relevancia: otros textos pueden responder bien.

In [ ]:
paths = real_paths(root)
corpus = jsonl(paths["corpus"])
queries = jsonl(paths["queries"])
qrels = json.loads(paths["qrels"].read_text(encoding="utf-8"))
assert len(corpus) == 48 and len(queries) == 10570
assert set(qrels) == {q["query_id"] for q in queries}
q = queries[0]
print("Pregunta original:", q["source_question_id"], q["query"])
print("Artículo:", q["source_title"], "| partición:", q["split"])
print("Pasaje de origen:", q["passage_id"])

#### **4. Cómo distinguen las arquitecturas sus decisiones**

- *Pipeline*: funciones en serie, sin elegir acciones en ejecución.
- *Workflow con ramas*: un programa selecciona caminos deterministas según observaciones.
- *Agente LLM*: un modelo propone acciones a partir del estado; un controlador externo comprueba contratos antes de ejecutarlas.

Los patrones básicos son *prompt chaining*, *routing*, paralelización, *evaluator–optimizer* y *orchestrator–workers*. La Semana 11 desarrolla este último con mayor profundidad.

Estado y actualización de un controlador:

$$
s_t=(q,t,O_t,C_t,B_t),\quad a_t=\pi(s_t),\quad s_{t+1}=F(s_t,a_t,o_{t+1}).
$$

La política puede ser una regla o un LLM. Un `while` no convierte por sí solo un sistema en agente autónomo.

#### **5. BM25, evidencia y compromisos**

En una colección de \(N\) pasajes, el BM25 simplificado usa:

$$
\operatorname{IDF}(w)=\log\left(1+\frac{N-df(w)+0.5}{df(w)+0.5}\right),
$$
$$
\operatorname{BM25}(q,d)=\sum_{w\in q}\operatorname{IDF}(w)
\frac{f(w,d)(k_1+1)}{f(w,d)+k_1(1-b+b|d|/\overline{|d|})}.
$$

Se emplea $k_1=1.5$, $b=0.75$. La herramienta utiliza un índice invertido, no cientos de millones de comparaciones texto a texto en Python.

In [ ]:
bm25 = BM25Tool(corpus)
assert len(bm25.passages) == 2067
rank = bm25.search(q["query"], 5)
for index, passage in enumerate(rank, start=1):
    print(index, passage["passage_id"], round(passage["score"], 3), passage["text"][:95])
print("Pasaje de origen anotado:", q["passage_id"])

#### **6. Política adaptativa y por qué no puede superar top-3**

Sea \(T(x)\) el conjunto de términos distintos de \(x\). Se define cobertura léxica:

\[
c(q,d_1)=\frac{|T(q)\cap T(d_1)|}{|T(q)|},\qquad
k(q)=\begin{cases}1&c(q,d_1)\ge\tau\\3&c(q,d_1)<\tau\end{cases}
\]

Para cada pregunta se cumple \(C_{A0}\subseteq C_B\subseteq C_{A1}\). Por tanto, `evidence_hit(A0) ≤ evidence_hit(B) ≤ evidence_hit(A1)` por construcción. **Esto no demuestra calidad de la respuesta.**

In [ ]:
tau_demostrativo = 0.45  # Solo para mostrar el mecanismo; NO es el umbral final calibrado.
a0 = run_workflow(q["query"], bm25, 1)
a1 = run_workflow(q["query"], bm25, 3)
b = run_adaptive(q["query"], bm25, tau_demostrativo)
first = bm25.search(q["query"], 1)[0]
print("Cobertura:", round(lexical_coverage(q["query"], first["text"]), 4))
for label, r in (("A0",a0),("A1",a1),("B",b)):
    print(label, "citas:", r["citations"], "llamadas:", r["tool_calls"])
assert set(a0["citations"]) <= set(b["citations"]) <= set(a1["citations"])

#### **7. Selección de umbral solo en calibración**

El código ensaya una rejilla de umbrales en 12 artículos de calibración. Conserva como objetivo que la pérdida absoluta frente a top-3 no supere dos puntos porcentuales en desarrollo; entre candidatos admisibles prioriza ahorro de palabras, penalizando una segunda búsqueda. Si ninguno satisface el objetivo, publica el compromiso en vez de ocultarlo.

La partición interna de SQuAD dev **no es el test oficial oculto**. La regla se fija una vez antes de interpretar los 36 artículos de evaluación.

In [ ]:
summary = evaluate_run(root, root / "Semana9" / "resultados")
print("Preguntas totales:", summary["sample_size"])
print("Preguntas de calibración:", summary["calibration_queries"])
print("Preguntas de evaluación:", summary["evaluation_queries"])
print("Umbral elegido en calibración:", summary["selected_threshold"])
print("Métricas de recuperación en EVALUACIÓN:")
print(json.dumps(summary["retrieval_evaluation"], ensure_ascii=False, indent=2))
print("Políticas en EVALUACIÓN:")
print(json.dumps(summary["evaluation_results"], ensure_ascii=False, indent=2))

#### **8. Métricas, errores y amenazas a la validez**

`Recall@k` indica si el pasaje de origen aparece dentro del top-k, y `MRR@10` pondera el rango \(r_q\) del primer pasaje de origen encontrado:

$$
R@k=\frac{1}{|Q|}\sum_q\mathbb{1}[r_q\le k],\qquad
MRR@10=\frac{1}{|Q|}\sum_q\frac{\mathbb{1}[r_q\le10]}{r_q}.
$$

Un `evidence_hit` no mide EM/F1 ni fidelidad factual. Un error puede deberse a paráfrasis, ruido, ambigüedad del pasaje original o falta de relevancia exhaustiva. Revisar casos individuales, nunca inventar explicaciones sin leerlos.

In [ ]:
import csv
with (root / "Semana9" / "resultados" / "runs.csv").open(encoding="utf-8", newline="") as fh:
    records = list(csv.DictReader(fh))
assert len(records) == 3 * 10570
rows_test = [r for r in records if r["split"] == "evaluation"]
failures = [r for r in rows_test if r["policy"] == "workflow_fixed_top3" and r["evidence_hit"] == "0"]
print("Fallos top-3 en evaluación:", len(failures))
questions = {q["query_id"]: q for q in queries}
for r in failures[:5]:
    item = questions[r["query_id"]]
    print(item["query_id"], item["query"], "pasaje de origen:", item["passage_id"])
    print("Recuperados:", r["citations"])

#### **9. Contratos del controlador de agente**

El controlador de `agent_loop.py` separa el modelo de las herramientas. Acepta únicamente `search_documents`, `read_passage` y `finish`; verifica argumentos y citas observadas. Rechaza acciones repetidas y finaliza por límites declarados. Estas garantías se comprueban mediante pruebas unitarias, **no son resultados de SQuAD**.

Importante: `completed_with_citations` significa que los identificadores estaban observados; no prueba que la respuesta esté semánticamente respaldada. No se inyectan ataques como datos del benchmark de preguntas.

In [ ]:
import unittest
suite = unittest.TestSuite()
for module in ("Semana9.test_agent_loop", "Semana9.test_agent_benchmark", "Semana9.test_prepare_real_data", "Semana9.test_agent_llm_eval"):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromName(module))
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()

#### **10. ReAct verdadero: solo si Qwen ejecuta acciones**

El piloto opcional compara **el mismo Qwen** en un workflow top-3 y en un agente que selecciona acciones. Ambos responden las mismas consultas de la partición de evaluación. Se calculan exact match y F1 contra las respuestas humanas originales; una cita válida se registra en una métrica distinta.

No se ejecuta por defecto para evitar descargar pesos durante la sesión. El docente puede ejecutarlo previamente con:

```bash
python -m Semana9.agent_llm_eval --n 100 --model-id Qwen/Qwen2.5-0.5B-Instruct
```

Si falta el modelo no se sustituye por una simulación. El piloto de 100 preguntas no reemplaza los 10.570 casos del benchmark BM25, ni autoriza generalización.

In [ ]:
from pathlib import Path
llm_path = root / "Semana9" / "resultados" / "llm_real" / "resumen_llm.json"
if llm_path.is_file():
    observed = json.loads(llm_path.read_text(encoding="utf-8"))
    print("Inferencia LLM realmente registrada:", observed["model_id"])
    print(json.dumps(observed["metrics"], ensure_ascii=False, indent=2))
else:
    print("NO EJECUTADO: piloto Qwen real. No mostrar resultados inventados.")

#### **11. Interpretación y preguntas para discusión**

Explica por qué B no puede recuperar evidencia fuera del top-3 de A1; identifica con números una pérdida o ahorro de contexto en evaluación; distingue un fallo de búsqueda de un fallo de respuesta; explica por qué `finish` puede ser estructuralmente válido y semánticamente incorrecto.

**Fuentes:** Rajpurkar et al. (2016), *SQuAD*; Yao et al. (2023), *ReAct*; Anthropic (2024), *Building Effective Agents*; Kapoor et al. (2024), *AI Agents That Matter*.